# Day 14 - GridSearchCV / RandomizedSearchCV + XGBoost


## 학습 목표
- GridSearchCV vs RandomizedSearchCV 비교
- XGBoost 하이퍼파라미터 튜닝


## 1. 데이터 준비


In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, accuracy_score
import xgboost as xgb
import time
import warnings
warnings.filterwarnings('ignore')

# 회귀용
data_reg = load_diabetes()
X_reg, y_reg = data_reg.data, data_reg.target
X_reg = StandardScaler().fit_transform(X_reg)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# 분류용
data_clf = load_breast_cancer()
X_clf, y_clf = data_clf.data, data_clf.target
X_clf = StandardScaler().fit_transform(X_clf)
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

print("Regression train:", X_tr_r.shape)
print("Classification train:", X_tr_c.shape)


## 2. GridSearchCV (XGBoost 회귀)


In [ ]:
param_grid = {
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [50, 100],
    'subsample': [0.8, 1.0]
}

model = xgb.XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)
grid = GridSearchCV(model, param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1)

start = time.time()
grid.fit(X_tr_r, y_tr_r)
elapsed = time.time() - start

print(f"GridSearch time: {elapsed:.1f}s")
print(f"Best params: {grid.best_params_}")
print(f"Best CV R2 : {grid.best_score_:.4f}")
print(f"Test R2    : {r2_score(y_te_r, grid.predict(X_te_r)):.4f}")


## 3. RandomizedSearchCV (XGBoost 분류)


In [ ]:
from scipy.stats import uniform, randint

param_dist = {
    'max_depth': randint(2, 8),
    'learning_rate': uniform(0.01, 0.2),
    'n_estimators': randint(50, 200),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4)
}

model = xgb.XGBClassifier(random_state=42, n_jobs=-1, verbosity=0, use_label_encoder=False, eval_metric='logloss')
rand = RandomizedSearchCV(model, param_dist, n_iter=20, cv=3, scoring='accuracy', n_jobs=-1, random_state=42, verbose=1)

start = time.time()
rand.fit(X_tr_c, y_tr_c)
elapsed = time.time() - start

print(f"RandomizedSearch time: {elapsed:.1f}s")
print(f"Best params: {rand.best_params_}")
print(f"Best CV Acc: {rand.best_score_:.4f}")
print(f"Test Acc   : {accuracy_score(y_te_c, rand.predict(X_te_c)):.4f}")
